# Deep Learning for NLP

**Deep Learning** is a type of Machine Learning that uses **Neural Networks with multiple layers** to learn patterns from data.

Deep Learning can be used in NLP tasks such as:
- Text Classification
- Sentiment Analysis
- Question Answering

In this lab, we will build a **deep learning model** to classify Yelp restaurant reviews as:

- **0 = Negative**
- **1 = Positive**

<img src="https://drive.google.com/uc?export=view&id=1rUaLO1wEOcUZ10Kwd0nUvS3jQzrqoOcI" width="550">
<img src="https://drive.google.com/uc?export=view&id=1XKPTKwxmIpP5cZIkDajlai6aI9jUX-C8" width="400">

## 1. Load the Yelp Dataset

Each row contains:

- a restaurant review
- its sentiment label


In [1]:
import pandas as pd

df = pd.read_csv("07-yelp-dataset.txt", sep="\t", names=["review", "label"])

print("Number of reviews:", len(df))
df.head()

Number of reviews: 1000


,review,label
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [2]:
print(df["label"].value_counts())

label
1    500
0    500
Name: count, dtype: int64


## 2. Preprocessing the Dataset

In [3]:
import re

def preprocess_text(text):

    # 1. Convert to lowercase
    text = text.lower()

    # 2. Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # 3. Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df["review"] = df["review"].apply(preprocess_text)
df.head()

,review,label
0,wow loved this place,1
1,crust is not good,0
2,not tasty and the texture was just nasty,0
3,stopped by during the late may bank holiday of...,1
4,the selection on the menu was great and so wer...,1


## 3. Split the Data

We divide the dataset into:

- **80% training data** — used to train the model
- **20% testing data** — used to evaluate the model

`stratify=y` keeps the proportion of positive and negative reviews similar in both sets.

In [4]:
from sklearn.model_selection import train_test_split

X = df["review"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))

Training reviews: 800
Testing reviews: 200


## 4. Convert Text to TF-IDF Vectors

A neural network cannot directly understand text.

**TF-IDF converts each review into a vector of numbers.**


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=1000, lowercase=True)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (800, 1000)
Testing TF-IDF shape: (200, 1000)


## 5. Convert the Data to PyTorch Tensors

PyTorch neural networks work with **tensors**.

In [6]:
import torch
import torch.nn as nn

# Convert TF-IDF data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train_tfidf.toarray())
X_test_tensor = torch.FloatTensor(X_test_tfidf.toarray())

y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)

## 6. Build a Simple Neural Network

Our network has:

1. **Input layer** — receives the TF-IDF features
2. **Hidden layer** — learns useful patterns
3. **ReLU activation** — adds non-linearity
4. **Output layer** — produces one value for binary classification

We do **not** put `Sigmoid` inside the model because `BCEWithLogitsLoss` applies it internally in a numerically stable way.

In [7]:
input_size = X_train_tensor.shape[1]
# torch.size([800, 1000])

model = nn.Sequential(
    nn.Linear(input_size, 64),   # Input layer
    nn.ReLU(),

    nn.Linear(64, 32),           # Hidden layer
    nn.ReLU(),

    nn.Linear(32, 1)             # Output layer
)

print(model)

Sequential(
  (0): Linear(in_features=1000, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=1, bias=True)
)


## 7. Choose the Loss Function and Optimizer

- **BCEWithLogitsLoss** is used for binary classification.
To measure the model’s mistakes.
- **Adam** is responsible for changing the model's weights so that the loss becomes smaller.

In [8]:
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

## 8. Train the Neural Network

One **epoch** means the model has seen the full training dataset once.

For every epoch:

1. Make predictions
2. Calculate the loss
3. Calculate gradients with backpropagation
4. Update the weights

In [9]:
epochs = 10

for epoch in range(epochs):
    model.train()

    # 1. Forward pass
    outputs = model(X_train_tensor)

    # 2. Calculate loss
    loss = loss_function(outputs, y_train_tensor)

    # 3. Backpropagation
    loss.backward()

    # 4. Update model weights
    optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}/{epochs} - Loss: {loss.item():.4f}")

Epoch 5/10 - Loss: 0.6385
Epoch 10/10 - Loss: 0.4106


## 9. Evaluate the Model

The model outputs raw scores called **logits**.

We apply `sigmoid` to convert them to probabilities between 0 and 1.

- Probability **≥ 0.5 → Positive (1)**
- Probability **< 0.5 → Negative (0)**

In [10]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model.eval()

with torch.no_grad():
    test = model(X_test_tensor)
    test_probabilities = torch.sigmoid(test)
    predictions = (test_probabilities >= 0.5).int().numpy().flatten()

accuracy = accuracy_score(y_test, predictions)

print(f"Test Accuracy: {accuracy:.3f}")

Test Accuracy: 0.725


## 10. Classification Report and Confusion Matrix

The classification report shows:

- **Precision**
- **Recall**
- **F1-score**

The confusion matrix shows how many reviews were classified correctly and incorrectly.

In [11]:
print("Classification Report:")
print(classification_report(y_test, predictions, target_names=["Negative", "Positive"]))

print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))

Classification Report:
              precision    recall  f1-score   support

    Negative       0.69      0.77      0.73        96
    Positive       0.76      0.68      0.72       104

    accuracy                           0.72       200
   macro avg       0.73      0.73      0.72       200
weighted avg       0.73      0.72      0.72       200

Confusion Matrix:
[[74 22]
 [33 71]]


## Task 1 – Represent Reviews Using Word2Vec

In the previous part of the lab, TF-IDF was used to convert each review into numerical features.
Now, use Word2Vec instead.

Requirements:
Tokenize the training and testing reviews.
Train a Word2Vec model using the training reviews.
Use:
vector_size = 200
window = 5
min_count = 1
sg = 1

Convert each review into a single vector.
Convert the resulting vectors into PyTorch tensors.


### 1. Tokenize the training and testing reviews

In [12]:
import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt_tab')

# X_train / X_test are the same pre-processed reviews that were used with TF-IDF
train_tokens = [word_tokenize(review) for review in X_train]
test_tokens = [word_tokenize(review) for review in X_test]

print('Training reviews:', len(train_tokens))
print('Testing reviews :', len(test_tokens))
print('Example review  :', X_train.iloc[0])
print('Example tokens  :', train_tokens[0])

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\kemo7\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Training reviews: 800
Testing reviews : 200
Example review  : the worst was the salmon sashimi
Example tokens  : ['the', 'worst', 'was', 'the', 'salmon', 'sashimi']


### 2. Train a Word2Vec model using the training reviews

The model is trained **only on the training reviews**, so nothing from the test set is seen during training.

In [13]:
import gensim

w2v_model = gensim.models.Word2Vec(train_tokens, vector_size=200, window=5, min_count=1, sg=1)

print('Vocabulary size:', len(w2v_model.wv))
print('Vector size    :', w2v_model.wv.vector_size)
print("Words similar to 'good':", w2v_model.wv.most_similar('good', topn=5))

Vocabulary size: 1800
Vector size    : 200
Words similar to 'good': [('the', 0.9991260766983032), ('on', 0.9991030693054199), ('food', 0.9990906119346619), ('were', 0.9990901350975037), ('was', 0.9990794062614441)]


### 3. Convert each review into a single vector

Word2Vec gives one vector for every **word**, but the neural network needs one vector for every **review**. So each review is represented by the **average** of the vectors of its words. Words of the test reviews that the model never saw in training are skipped, and a review with no known words becomes a vector of zeros.

In [14]:
import numpy as np

def review_to_vector(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.wv.vector_size)
    return np.mean(vectors, axis=0)

X_train_w2v = np.array([review_to_vector(tokens, w2v_model) for tokens in train_tokens])
X_test_w2v = np.array([review_to_vector(tokens, w2v_model) for tokens in test_tokens])

print('Training Word2Vec shape:', X_train_w2v.shape)
print('Testing Word2Vec shape :', X_test_w2v.shape)

Training Word2Vec shape: (800, 200)
Testing Word2Vec shape : (200, 200)


### 4. Convert the resulting vectors into PyTorch tensors

In [15]:
X_train_w2v_tensor = torch.FloatTensor(X_train_w2v)
X_test_w2v_tensor = torch.FloatTensor(X_test_w2v)

# the labels are the same as before
y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)

print('X_train tensor:', X_train_w2v_tensor.shape)
print('X_test tensor :', X_test_w2v_tensor.shape)
print('y_train tensor:', y_train_tensor.shape)

X_train tensor: torch.Size([800, 200])
X_test tensor : torch.Size([200, 200])
y_train tensor: torch.Size([800, 1])


## Task 2 – Build a Different Neural Network

Use the Word2Vec vectors from Task 1 as the input to a new neural network.
Your network must contain:
An input layer
Three hidden layers
ReLU activation functions
One output neuron
Use the following architecture:

Input
   
   ↓

128 neurons
   
   ↓

64 neurons
   
   ↓

32 neurons
   
   ↓

1 output neuron

Train the model using:
, BCEWithLogitsLoss
, Adam optimizer
, Learning rate = 0.001
, 15 epochs.
Then calculate the test accuracy.

### 1. Build the neural network

In [16]:
torch.manual_seed(42)   # so the results are the same every time the cell is run

w2v_input_size = X_train_w2v_tensor.shape[1]   # 200

w2v_model_nn = nn.Sequential(
    nn.Linear(w2v_input_size, 128),   # Input layer -> hidden layer 1 (128 neurons)
    nn.ReLU(),

    nn.Linear(128, 64),               # Hidden layer 2 (64 neurons)
    nn.ReLU(),

    nn.Linear(64, 32),                # Hidden layer 3 (32 neurons)
    nn.ReLU(),

    nn.Linear(32, 1)                  # Output layer (1 neuron)
)

print(w2v_model_nn)

Sequential(
  (0): Linear(in_features=200, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=64, bias=True)
  (3): ReLU()
  (4): Linear(in_features=64, out_features=32, bias=True)
  (5): ReLU()
  (6): Linear(in_features=32, out_features=1, bias=True)
)


### 2. Train the model (BCEWithLogitsLoss, Adam, learning rate = 0.001, 15 epochs)

In [17]:
w2v_loss_function = nn.BCEWithLogitsLoss()
w2v_optimizer = torch.optim.Adam(w2v_model_nn.parameters(), lr=0.001)

epochs = 15

for epoch in range(epochs):
    w2v_model_nn.train()

    # clear the gradients of the previous epoch
    w2v_optimizer.zero_grad()

    # 1. Forward pass
    outputs = w2v_model_nn(X_train_w2v_tensor)

    # 2. Calculate loss
    loss = w2v_loss_function(outputs, y_train_tensor)

    # 3. Backpropagation
    loss.backward()

    # 4. Update model weights
    w2v_optimizer.step()

    print(f"Epoch {epoch + 1}/{epochs} - Loss: {loss.item():.4f}")

Epoch 1/15 - Loss: 0.6964
Epoch 2/15 - Loss: 0.6959
Epoch 3/15 - Loss: 0.6955
Epoch 4/15 - Loss: 0.6951
Epoch 5/15 - Loss: 0.6948
Epoch 6/15 - Loss: 0.6945
Epoch 7/15 - Loss: 0.6942
Epoch 8/15 - Loss: 0.6939
Epoch 9/15 - Loss: 0.6937
Epoch 10/15 - Loss: 0.6935
Epoch 11/15 - Loss: 0.6934


Epoch 12/15 - Loss: 0.6933
Epoch 13/15 - Loss: 0.6933


Epoch 14/15 - Loss: 0.6933
Epoch 15/15 - Loss: 0.6933


### 3. Calculate the test accuracy

In [18]:
w2v_model_nn.eval()

with torch.no_grad():
    test_logits = w2v_model_nn(X_test_w2v_tensor)
    test_probabilities = torch.sigmoid(test_logits)
    w2v_predictions = (test_probabilities >= 0.5).int().numpy().flatten()

w2v_accuracy = accuracy_score(y_test, w2v_predictions)

print(f"Test Accuracy (Word2Vec + 3 hidden layers): {w2v_accuracy:.3f}")
print(f"Test Accuracy (TF-IDF model from the lab) : {accuracy:.3f}")

Test Accuracy (Word2Vec + 3 hidden layers): 0.480
Test Accuracy (TF-IDF model from the lab) : 0.725


In [19]:
print("Classification Report:")
print(classification_report(y_test, w2v_predictions, target_names=["Negative", "Positive"], zero_division=0))

print("Confusion Matrix:")
print(confusion_matrix(y_test, w2v_predictions))

Classification Report:
              precision    recall  f1-score   support

    Negative       0.48      1.00      0.65        96
    Positive       0.00      0.00      0.00       104

    accuracy                           0.48       200
   macro avg       0.24      0.50      0.32       200
weighted avg       0.23      0.48      0.31       200

Confusion Matrix:
[[ 96   0]
 [104   0]]


### Why is the accuracy low?

With the settings required in the task the accuracy is close to **50%**, which is the same as guessing, and the loss stays near **0.693** (= ln 2, the loss of a model that always answers 50/50). The network is not wrong, it just did not have a chance to learn:

- **Word2Vec had very little data.** It was trained on only 800 short reviews with the default 5 epochs, so the word vectors are still close to their random starting values, and after averaging them almost every review gets a very similar vector.
- **The network took only 15 small steps.** The whole training set is used as one batch, so 15 epochs = 15 weight updates, with a small learning rate of 0.001.

### Extra: the same network with better training

To check this explanation, the cell below keeps **exactly the same network, loss, optimizer, learning rate and number of epochs**, and only changes two things: Word2Vec is trained for 50 epochs instead of 5, and the network is trained with mini-batches of 32 reviews (25 weight updates per epoch instead of 1).

In [20]:
from torch.utils.data import TensorDataset, DataLoader

# Word2Vec with the same parameters, but trained for more epochs
w2v_model_50 = gensim.models.Word2Vec(train_tokens, vector_size=200, window=5, min_count=1, sg=1,
                                      epochs=50, seed=42, workers=1)

X_train_50 = torch.FloatTensor(np.array([review_to_vector(t, w2v_model_50) for t in train_tokens]))
X_test_50 = torch.FloatTensor(np.array([review_to_vector(t, w2v_model_50) for t in test_tokens]))

torch.manual_seed(42)

extra_model = nn.Sequential(
    nn.Linear(200, 128), nn.ReLU(),
    nn.Linear(128, 64), nn.ReLU(),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, 1)
)

extra_loss_function = nn.BCEWithLogitsLoss()
extra_optimizer = torch.optim.Adam(extra_model.parameters(), lr=0.001)

loader = DataLoader(TensorDataset(X_train_50, y_train_tensor), batch_size=32, shuffle=True)

for epoch in range(15):
    extra_model.train()
    total_loss = 0
    for X_batch, y_batch in loader:
        extra_optimizer.zero_grad()
        loss = extra_loss_function(extra_model(X_batch), y_batch)
        loss.backward()
        extra_optimizer.step()
        total_loss += loss.item() * len(X_batch)

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}/15 - Loss: {total_loss / len(X_train_50):.4f}")

extra_model.eval()
with torch.no_grad():
    extra_predictions = (torch.sigmoid(extra_model(X_test_50)) >= 0.5).int().numpy().flatten()

extra_accuracy = accuracy_score(y_test, extra_predictions)
print(f"\nTest Accuracy (Word2Vec 50 epochs + mini-batches): {extra_accuracy:.3f}")

Epoch 5/15 - Loss: 0.5863


Epoch 10/15 - Loss: 0.5215


Epoch 15/15 - Loss: 0.4582

Test Accuracy (Word2Vec 50 epochs + mini-batches): 0.650


In [21]:
print('========== Results ==========')
print(f'TF-IDF + 2 hidden layers (lab example)          : {accuracy:.3f}')
print(f'Word2Vec + 3 hidden layers (Task 2)             : {w2v_accuracy:.3f}')
print(f'Word2Vec + 3 hidden layers (extra, more training): {extra_accuracy:.3f}')

========== Results ==========
TF-IDF + 2 hidden layers (lab example)          : 0.725
Word2Vec + 3 hidden layers (Task 2)             : 0.480
Word2Vec + 3 hidden layers (extra, more training): 0.650


_____________________________________________________________________________________________

# What I Learned from Lab 6

In this lab I learned how to build a **deep learning model** for text classification with PyTorch, and I compared two ways of representing the reviews: **TF-IDF** and **Word2Vec**.

### Building a Neural Network for Text
- A neural network cannot read text, so every review must first become a vector of numbers. In the lab example this was done with **TF-IDF** (1,000 features per review).
- PyTorch works with **tensors**, so the vectors are converted with `torch.FloatTensor`, and the labels are reshaped to one column with `.reshape(-1, 1)`.
- `nn.Sequential` stacks the layers: `nn.Linear` layers with **ReLU** between them to add non-linearity, and one output neuron for binary classification.
- **BCEWithLogitsLoss** measures the mistakes of a binary classifier and already contains the sigmoid, so the model does not need a `Sigmoid` layer. **Adam** updates the weights to make the loss smaller.
- The training loop has four steps: forward pass, calculate the loss, backpropagation (`loss.backward()`) and update the weights (`optimizer.step()`). I also call `optimizer.zero_grad()` first, because PyTorch adds the new gradients to the old ones if they are not cleared.
- In evaluation I use `model.eval()` and `torch.no_grad()`, then `sigmoid` turns the logits into probabilities (≥ 0.5 → positive).

### Task 1: Representing Reviews Using Word2Vec
- I tokenized the 800 training and 200 testing reviews with `word_tokenize`.
- I trained a skip-gram Word2Vec model (`vector_size=200`, `window=5`, `min_count=1`, `sg=1`) **only on the training reviews**, so the test set stays unseen. The vocabulary had **1,800 words**.
- Word2Vec gives a vector per word, so I represented each review by the **average** of its word vectors. This gives a (800, 200) training matrix and a (200, 200) testing matrix.
- Test words that are not in the vocabulary are skipped, and a review with no known words becomes a vector of zeros.

### Task 2: A Different Neural Network
- I built a network with three hidden layers (200 → 128 → 64 → 32 → 1) and trained it with BCEWithLogitsLoss, Adam, learning rate 0.001 and 15 epochs.
- The test accuracy was only **0.480**, compared to **0.725** for the TF-IDF model. The loss stayed at about **0.693**, which is ln 2, the loss of a model that is just guessing. The confusion matrix showed that the model predicted *negative* for every review.
- I learned to read these signs: a loss that does not move and a model that predicts one class mean that the model **did not learn**, not that the code has an error.
- There were two reasons. Word2Vec was trained on only 800 short reviews for 5 epochs, so all the word vectors were still very similar (the words most similar to `good` were `the`, `on` and `food`, all with a similarity of 0.999). And training on the full batch means 15 epochs is only 15 small weight updates.
- When I kept the same network and settings but trained Word2Vec for 50 epochs and used mini-batches of 32, the accuracy went up to **0.650**. This confirmed the explanation.

### TF-IDF vs Word2Vec
- A bigger network is not automatically better. The deeper Word2Vec network lost against the smaller TF-IDF network.
- **Word2Vec needs a lot of text** to learn good vectors. With only 1,000 reviews, TF-IDF works better because it does not need to learn anything about the words.
- Averaging the word vectors also loses information such as word order, so "not good" and "good" get close vectors.

**Conclusion:** The quality of the text representation matters more than the size of the network. On a small dataset a simple representation like TF-IDF can beat word embeddings, and word embeddings become useful when they are trained on much more data (or when pre-trained vectors are used).